In [1]:
import json
import statistics
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd().parent
DB_PATH = REPO_ROOT / "database" / "CSCAF_public.db"
OUTPUT_FOLDER = REPO_ROOT / "methodology" / "output"
PUBLISHED_THRESHOLD = 0.671

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

assert DB_PATH.exists(), "run the notebooks of data_load first"

from cscaf.storage import Storage
from cscaf.study import Study

study = Study.from_config()
TAU = study.tau
RUN = study.run_id
storage = Storage.of(study)
print("database          :", DB_PATH.relative_to(REPO_ROOT))
print("run               :", RUN)
print("tau (config.yaml) :", TAU)
print("published         :", PUBLISHED_THRESHOLD)

database          : database\CSCAF_public.db
run               : sat-12l-sm_all-mpnet-base-v2_414c4542
tau (config.yaml) : 0.585
published         : 0.671


In [2]:
curricula = storage.curricula.keys()
statements = storage.framework.count_statements(RUN)
advertisements = storage.advertisements.count()
rows = storage.similarities.count_curriculum_best_matches(RUN)
assert rows == len(curricula) * statements, rows
rows = storage.similarities.count_advertisement_best_matches(RUN)
assert rows == advertisements * statements, rows
{"curricula": len(curricula), "statements": statements, "advertisements": advertisements}

{'curricula': 27, 'statements': 150, 'advertisements': 5136}

In [3]:
def effect(threshold):
    covered = storage.similarities.statements_covered_per_curriculum(RUN, threshold)
    per_curriculum = {key: covered.get(key, 0) for key in curricula}
    with_statement = storage.similarities.advertisements_with_statement(RUN, threshold)
    without = advertisements - with_statement
    return {"threshold": threshold,
            "curriculum_statements_median": statistics.median(per_curriculum.values()),
            "curriculum_statements_min": min(per_curriculum.values()),
            "curriculum_statements_max": max(per_curriculum.values()),
            "advertisements_without_statement": without,
            "advertisements_without_statement_share": without / advertisements,
            "per_curriculum": per_curriculum}


results = {"tau": effect(TAU), "published": effect(PUBLISHED_THRESHOLD)}
print(f"{'curriculum':14s} {'tau':>5s} {'0.671':>6s}")
for key in curricula:
    print(f"{key:14s} {results['tau']['per_curriculum'][key]:5d} {results['published']['per_curriculum'][key]:6d}")
for name, r in results.items():
    print(f"\n{name:9s} threshold {r['threshold']}: curricula median {r['curriculum_statements_median']} "
          f"(min {r['curriculum_statements_min']}, max {r['curriculum_statements_max']}) of {statements} statements; "
          f"advertisements without any statement {r['advertisements_without_statement']} of {advertisements} "
          f"({r['advertisements_without_statement_share']:.1%})")

curriculum       tau  0.671
at_ustp           75     24
be_kuleuven       80     32
bg_nbu            88     52
cy_ouc            44      7
cz_ctu            45     14
de_hdbw           90     36
dk_aau            90     33
ee_taltech        96     41
es_uah            89     33
fi_jamk           71     25
fr_ubs            40     13
gr_ihu            85     39
hr_tvz           102     49
hu_nke           113     80
ie_mtu            73     29
it_unipi          58     16
lt_vilniustech    39      9
lu_unilu          95     32
lv_lu             63     19
mt_um             91     32
nl_uva            25      6
pl_prz            56     16
pt_ipleiria       41      8
ro_utcn           72     27
se_kth            93     27
si_fis           111     49
sk_tuke           21      6

tau       threshold 0.585: curricula median 75 (min 21, max 113) of 150 statements; advertisements without any statement 144 of 5136 (2.8%)

published threshold 0.671: curricula median 27 (min 6, max 80) of 150 stat

In [4]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
output = OUTPUT_FOLDER / "03_text_processing.json"
record = {
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "paper_section": "3.3",
    "run": RUN,
    "tau": TAU,
    "published_threshold": PUBLISHED_THRESHOLD,
    "curricula": len(curricula),
    "statements": statements,
    "advertisements": advertisements,
    "effect": results,
}
output.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
print("written to", output.relative_to(REPO_ROOT))

written to methodology\output\03_text_processing.json
